### Competition EDA

In [53]:
import re
import pandas as pd
import statistics
from tqdm import tqdm

tqdm.pandas()


In [54]:
import sys
sys.path.append("..")

from src.solvers.bit_manipulation import BitManipulationSolver
from src.solvers.equations import EnsembleEquationsSolver
from src.solvers.encryption import EncryptionSolver
from src.solvers.unit_conversion import UnitConversionSolver
from src.solvers.gravitational import GravitationalSolver
from src.solvers.numeral_system import NumeralSystemSolver

In [55]:
data = pd.read_csv("../data/raw/train.csv")

In [56]:
data

,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret
...,...,...,...
9495,ffce9e31,"In Alice's Wonderland, a secret bit manipulati...",01100110
9496,ffd5bada,"In Alice's Wonderland, a secret unit conversio...",32.45
9497,ffd89354,"In Alice's Wonderland, secret encryption rules...",student sees the curious mirror
9498,ffdfb678,"In Alice's Wonderland, secret encryption rules...",the curious mouse creates


In [57]:
data["prompt_eda"] = data.prompt.str.split('.').apply(lambda x: x[0])

In [58]:
print(data.prompt_eda.unique())

<ArrowStringArray>
['In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers',
                       'In Alice's Wonderland, secret encryption rules are used on text',
 'In Alice's Wonderland, numbers are secretly converted into a different numeral system',
            'In Alice's Wonderland, a secret unit conversion is applied to measurements',
           'In Alice's Wonderland, the gravitational constant has been secretly changed',
   'In Alice's Wonderland, a secret set of transformation rules is applied to equations']
Length: 6, dtype: str


In [59]:
task_classes = {
    "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers":  "bit manipulation",
    "In Alice's Wonderland, secret encryption rules are used on text": "encryption",
    "In Alice's Wonderland, numbers are secretly converted into a different numeral system": "conversion to diff numeral system",
    "In Alice's Wonderland, a secret unit conversion is applied to measurements": "unit conversion",
    "In Alice's Wonderland, the gravitational constant has been secretly changed": "gravitational",
    "In Alice's Wonderland, a secret set of transformation rules is applied to equations": "equations transformation"
}

In [60]:
data["label"] = data.prompt_eda.map(task_classes)
data["label"].value_counts()

label
bit manipulation                     1602
gravitational                        1597
unit conversion                      1594
encryption                           1576
conversion to diff numeral system    1576
equations transformation             1555
Name: count, dtype: int64

In [61]:
### Just check data

In [62]:
data.iloc[2].prompt

"In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nucoov pwgtfyoqg vorq yrjjoe -> queen discovers near valley\npqrsfv pqorzg wvgwpo trgbjo -> dragon dreams inside castle\ngbcpovb tqorbog bxo zrswtrj pffq -> student creates the magical door\nbxo sfjpov pqrsfv dfjjfig -> the golden dragon follows\nnqwvtogg qorpg bxo zegboqwfcg gotqob -> princess reads the mysterious secret\nNow, decrypt the following text: trb wzrswvog hffk"

In [63]:
data[data.label == "equations transformation"].sample(1).prompt.to_list()

["In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n88+71 = 501\n76$06 = 9104\n99+56 = 461\n86+56 = 331\n93$23 = 7421\nNow, determine the result for: 88$13"]

In [64]:
data[data.label == "bit manipulation"].sample(1).prompt.to_list()

["In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n11101111 -> 00011011\n01111010 -> 00010100\n01011110 -> 00010100\n10111110 -> 00101101\n11101010 -> 00010000\n01110001 -> 00000000\n00000010 -> 00000000\n11100011 -> 00000000\n11101100 -> 00011001\n11011001 -> 00110010\n\nNow, determine the output for: 11000101"]

In [65]:
data["label_format"] = data.prompt.str.split("\n").apply(lambda x: x[-1])

In [66]:
data.groupby("label")["label_format"].value_counts().to_dict()

{('bit manipulation', 'Now, determine the output for: 01010101'): 13,
 ('bit manipulation', 'Now, determine the output for: 11110101'): 13,
 ('bit manipulation', 'Now, determine the output for: 11001101'): 12,
 ('bit manipulation', 'Now, determine the output for: 10000001'): 12,
 ('bit manipulation', 'Now, determine the output for: 11001000'): 12,
 ('bit manipulation', 'Now, determine the output for: 10101001'): 12,
 ('bit manipulation', 'Now, determine the output for: 10001001'): 12,
 ('bit manipulation', 'Now, determine the output for: 01111110'): 11,
 ('bit manipulation', 'Now, determine the output for: 11101101'): 11,
 ('bit manipulation', 'Now, determine the output for: 11111010'): 11,
 ('bit manipulation', 'Now, determine the output for: 11100110'): 11,
 ('bit manipulation', 'Now, determine the output for: 11000110'): 11,
 ('bit manipulation', 'Now, determine the output for: 00100110'): 11,
 ('bit manipulation', 'Now, determine the output for: 00110000'): 10,
 ('bit manipulation'

In [67]:
def extract_template(text):
    if not isinstance(text, str):
        return str(text)
        
    text = text.strip()
    
    # "Now, determine the output for: <TARGET>"
    if ':' in text:
        return re.sub(r':\s*.*$', ': <TARGET>', text)
        
    # "If <NUM> @ <NUM> = <NUM>, what is X?"
    text = re.sub(r'\b\d+\b', '<NUM>', text)
    
    return text

data["pattern"] = data["label_format"].apply(extract_template)

patterns_summary = data.groupby("label")["pattern"].value_counts().to_frame("count").reset_index()

for label in patterns_summary['label'].unique():
    print(f"\n=== {label} ===")
    subset = patterns_summary[patterns_summary['label'] == label]
    for _, row in subset.iterrows():
        print(f"{row['count']:>4} | {row['pattern']}")


=== bit manipulation ===
1602 | Now, determine the output for: <TARGET>

=== conversion to diff numeral system ===
1576 | Now, write the number <NUM> in the Wonderland numeral system.

=== encryption ===
1576 | Now, decrypt the following text: <TARGET>

=== equations transformation ===
1555 | Now, determine the result for: <TARGET>

=== gravitational ===
  25 | Now, determine the falling distance for t = <NUM>.32s given d = <NUM>.<NUM>*g*t^<NUM>.
  24 | Now, determine the falling distance for t = <NUM>.82s given d = <NUM>.<NUM>*g*t^<NUM>.
  24 | Now, determine the falling distance for t = <NUM>.72s given d = <NUM>.<NUM>*g*t^<NUM>.
  23 | Now, determine the falling distance for t = <NUM>.79s given d = <NUM>.<NUM>*g*t^<NUM>.
  23 | Now, determine the falling distance for t = <NUM>.87s given d = <NUM>.<NUM>*g*t^<NUM>.
  22 | Now, determine the falling distance for t = <NUM>.45s given d = <NUM>.<NUM>*g*t^<NUM>.
  22 | Now, determine the falling distance for t = <NUM>.57s given d = <NUM>.<

### conversion to diff numeral system

In [68]:
numeral_df = data[data['label'] == 'conversion to diff numeral system'].copy()

solver = NumeralSystemSolver()

numeral_df['generated_cot'] = numeral_df['prompt'].apply(solver.generate_cot)

numeral_df['computed_answer'] = numeral_df['generated_cot'].apply(solver.extract_answer)

numeral_df['is_correct'] = numeral_df['computed_answer'].astype(str).str.strip() == numeral_df['answer'].astype(str).str.strip()

accuracy = numeral_df['is_correct'].mean()
print(f"Accuracy by '{numeral_df['label'].iloc[0]}': {accuracy * 100:.2f}%")

Accuracy by 'conversion to diff numeral system': 100.00%


In [69]:
numeral_df

,id,prompt,answer,prompt_eda,label,label_format,pattern,generated_cot,computed_answer,is_correct
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 38 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,XXXVIII,True
14,00600e6e,"In Alice's Wonderland, numbers are secretly co...",LXVII,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 67 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,LXVII,True
30,00d9f682,"In Alice's Wonderland, numbers are secretly co...",C,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 100 in the Wonderland nu...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,C,True
36,0106eb4a,"In Alice's Wonderland, numbers are secretly co...",LXXXIV,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 84 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,LXXXIV,True
37,0122d53a,"In Alice's Wonderland, numbers are secretly co...",LI,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 51 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,LI,True
...,...,...,...,...,...,...,...,...,...,...
9476,ff5cb472,"In Alice's Wonderland, numbers are secretly co...",V,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 5 in the Wonderland nume...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,V,True
9477,ff5f4ff2,"In Alice's Wonderland, numbers are secretly co...",LI,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 51 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,LI,True
9478,ff612478,"In Alice's Wonderland, numbers are secretly co...",XXI,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 21 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,XXI,True
9479,ff650fc3,"In Alice's Wonderland, numbers are secretly co...",XXXVI,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 36 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,XXXVI,True


### unit conversion

In [70]:
unit_df = data[data['label'] == 'unit conversion'].copy()

solver = UnitConversionSolver()

unit_df['generated_cot'] = unit_df['prompt'].apply(solver.generate_cot)

unit_df['computed_answer'] = unit_df['generated_cot'].apply(solver.extract_answer)

unit_df['is_correct'] = unit_df['computed_answer'].astype(str).str.strip() == unit_df['answer'].astype(str).str.strip()

accuracy = unit_df['is_correct'].mean()
print(f"Accuracy by '{unit_df['label'].iloc[0]}': {accuracy * 100:.2f}%")

Accuracy by 'unit conversion': 90.72%


In [71]:

errors_df = unit_df[~unit_df['is_correct']]
for idx, row in errors_df.sample(3).iterrows():
    print(f"=== ID: {row['id']} ===")
    print(f"answer:  '{row['answer']}'")
    print(f"Computed:'{row['computed_answer']}'")
    print(f"Prompt: {row['prompt'][-80:]}\n")

=== ID: c7e03ea1 ===
answer:  '18.55'
Computed:'18.56'
Prompt: omes 29.97
30.02 m becomes 24.44
Now, convert the following measurement: 22.79 m

=== ID: 6dee6c75 ===
answer:  '52.91'
Computed:'52.90'
Prompt: omes 33.18
29.06 m becomes 37.98
Now, convert the following measurement: 40.48 m

=== ID: 82d14e7a ===
answer:  '45.53'
Computed:'45.52'
Prompt: comes 38.60
30.01 m becomes 27.49
Now, convert the following measurement: 49.7 m



### gravitational

In [72]:
grav_df = data[data['label'] == 'gravitational'].copy()

solver = GravitationalSolver()

grav_df['generated_cot'] = grav_df['prompt'].apply(solver.generate_cot)

grav_df['computed_answer'] = grav_df['generated_cot'].apply(solver.extract_answer)

grav_df['is_correct'] = grav_df['computed_answer'].astype(str).str.strip() == grav_df['answer'].astype(str).str.strip()

accuracy = grav_df['is_correct'].mean()
print(f"Accuracy by '{grav_df['label'].iloc[0]}': {accuracy * 100:.2f}%")

Accuracy by 'gravitational': 77.46%


In [73]:

errors_df = grav_df[~grav_df['is_correct']]
for idx, row in errors_df.sample(3).iterrows():
    print(f"=== ID: {row['id']} ===")
    print(f"answer:  '{row['answer']}'")
    print(f"Computed:'{row['computed_answer']}'")
    print(f"Prompt: {row['prompt'][-80:]}\n")

=== ID: 479e86b4 ===
answer:  '20.1'
Computed:'20.10'
Prompt: = 52.77 m
Now, determine the falling distance for t = 2.16s given d = 0.5*g*t^2.

=== ID: a4f165c1 ===
answer:  '61.38'
Computed:'61.35'
Prompt:  = 6.51 m
Now, determine the falling distance for t = 3.53s given d = 0.5*g*t^2.

=== ID: d7032308 ===
answer:  '186.4'
Computed:'186.40'
Prompt:  183.01 m
Now, determine the falling distance for t = 4.38s given d = 0.5*g*t^2.



### encryption

In [74]:
enc_df = data[data['label'] == 'encryption'].copy()

global_vocab = set()
for prompt in enc_df['prompt']:
    lines = [l.strip() for l in prompt.lower().splitlines() if "->" in l]
    for line in lines:
        plain = line.split("->", 1)[1]
        words = re.sub(r"[^a-z\s]", "", plain).split()
        global_vocab.update(words)

#for ans in enc_df['answer']:
#    if isinstance(ans, str):
#         global_vocab.update(re.sub(r"[^a-z\s]", "", ans.lower()).split())

print(f"Vocabulary from prompts: {len(global_vocab)}\n")

solver = EncryptionSolver(vocabulary=global_vocab)

enc_df['generated_cot'] = enc_df['prompt'].apply(lambda x: solver.generate_cot(x))
enc_df['computed_answer'] = enc_df['generated_cot'].apply(solver.extract_answer)

failed_mask = enc_df['computed_answer'].isna()
print(f"Fail on first run: {failed_mask.sum()} rows {len(enc_df)}\n")

if failed_mask.sum() > 0:
    def solve_with_fallback(row):
        return solver.generate_cot(row['prompt'], answer_hint=row['answer'])

    enc_df.loc[failed_mask, 'generated_cot'] = enc_df[failed_mask].apply(solve_with_fallback, axis=1)
    
    enc_df.loc[failed_mask, 'computed_answer'] = enc_df.loc[failed_mask, 'generated_cot'].apply(solver.extract_answer)

enc_df['is_correct'] = enc_df['computed_answer'] == enc_df['answer'].astype(str).str.lower().str.strip()
final_accuracy = enc_df['is_correct'].mean() * 100

print(f"Final Accuracy: {final_accuracy:.2f}%")

Vocabulary from prompts: 77

Fail on first run: 0 rows 1576

Final Accuracy: 99.05%


### 

In [75]:
from pandarallel import pandarallel

pandarallel.initialize(nb_workers=24, progress_bar=True)


INFO: Pandarallel will run on 24 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [76]:
eq_df = data[data['label'] == 'equations transformation'].copy()

In [77]:
solver = EnsembleEquationsSolver()

eq_df['generated_cot'] = eq_df['prompt'].parallel_apply(solver.generate_cot)
eq_df['computed_answer'] = eq_df['generated_cot'].parallel_apply(solver.extract_answer)

# Если answer является строкой (символы или числа)
eq_df['is_correct'] = eq_df['computed_answer'] == eq_df['answer'].astype(str).str.strip()

accuracy = eq_df['is_correct'].mean()
print(f"Accuracy (Unified Solver): {accuracy * 100:.2f}%")

Accuracy (Unified Solver): 40.19%


In [78]:
# Фильтруем строки с ошибками (nan) и смотрим, на чем именно падает алгоритм
errors = eq_df[eq_df['computed_answer'] == 'nan']
if not errors.empty:
    print("\n--- ПРИМЕРЫ ОШИБОК ПАРСИНГА ИЛИ ГЕНЕРАЦИИ ---")
    print(errors[['prompt', 'generated_cot']].head(20).to_string())

# Если алгоритм выдает ответ, но он не совпадает с таргетом:
wrong_answers = eq_df[(eq_df['computed_answer'] != 'nan') & (~eq_df['is_correct'])]
if not wrong_answers.empty:
    print("\n--- ПРИМЕРЫ НЕВЕРНЫХ РЕШЕНИЙ ---")
    print(wrong_answers[['prompt', 'computed_answer', 'answer']].sample(20).to_string())


--- ПРИМЕРЫ НЕВЕРНЫХ РЕШЕНИЙ ---
                                                                                                                                                                                                                       prompt computed_answer answer
3368  In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n{{-(( = &&\n((-## = -{{\n[@+|& = [@|&\n|)*[{ = /&"[\n[#+#" = [##"\nNow, determine the result for: {@*|{            {@|{   /(#)
8185      In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n]]-|' = "%\n]"-%] = "\n$"*?$ = "|@`\n'|*#` = ##$\n'?+]% = |||\nNow, determine the result for: ?`*]%            ?`]%   %"|@
5696                           In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n'{*\$ = |"||\n||*:" = |){:\n"|*$> = ^$':\nNow, determine the result for: |'+$'          

In [79]:
errors_df = eq_df[~eq_df['is_correct']]
for idx, row in errors_df.sample(10).iterrows():
    print(f"=== ID: {row['id']} ===")
    print(f"answer:  '{row['answer']}'")
    print(f"Computed:'{row['computed_answer']}'")
    print(f"Prompt: {row['prompt'][-110:]}\n")

=== ID: 680b66f2 ===
answer:  '}$'
Computed:'!<<\'
Prompt: equations. Below are a few examples:
<{*^& = ^&<{
^|-[$ = ^<
[}*|< = |<[}
Now, determine the result for: !<+<\

=== ID: 58fed63a ===
answer:  '5'
Computed:'-82'
Prompt: to equations. Below are a few examples:
68*91 = 9168
06-65 = 4
86-72 = 41
Now, determine the result for: 11-39

=== ID: 5144897d ===
answer:  '0'
Computed:'1'
Prompt: . Below are a few examples:
99-66 = 6533
13:92 = 1
49|31 = 80
76|65 = 141
Now, determine the result for: 90:15

=== ID: 0bb591e6 ===
answer:  '%?<'
Computed:'"$%%'
Prompt: ions. Below are a few examples:
)(-)( = "
&(-(( = <
$(-(% = )<
<%-%$ = ??
Now, determine the result for: "$+%%

=== ID: 9935aa11 ===
answer:  '-`#'
Computed:')<<`'
Prompt: quations. Below are a few examples:
?@*?\ = ?\?@
(#*>< = ><(#
`\+)( = \##
Now, determine the result for: <`-)<

=== ID: cca882b8 ===
answer:  '>)<|'
Computed:'>)!{'
Prompt:  equations. Below are a few examples:
>{`!{ = >{!#
{|$|| = '<!
[&${& = <'
Now, determine

In [87]:
eq_df[eq_df.is_correct].generated_cot.to_list()

["The examples provided do not behave like mathematical equations. The operator is acting as a string manipulation function.\nBy tracking how the characters move from the left side of the equation to the right side, we can determine the exact operation.\nThe consistent pattern across all examples is to concatenate the two strings together from left to right.\n\nNow, let's process the target strings '\\(' and '[#' using this exact logic.\n- Following the rule, the new sequence of characters becomes '\\([#'.\n\nFinal answer: \\([#",
 "First, let's analyze the underlying pattern in the provided examples.\nThe standard mathematical operators are being used as placeholders for a hidden, multi-step rule.\nBy observing the relationship between the inputs and outputs, the consistent sequence of operations is:\nStep 1: We must reverse the digits of each number.\nStep 2: Next, we multiply the numbers.\nStep 3: Finally, we reverse the digits of the final computed result.\n\nNow, let's apply this 

In [80]:
d = eq_df.sample(3)

In [81]:
d.prompt.to_list()

["In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n06-45 = 6\n49*58 = 1997\n22*17 = 3651\n42-65 = -23\n69*08 = 1867\nNow, determine the result for: 24*88",
 "In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n65+83 = 8365\n62-23 = -6\n69*72 = 2952\nNow, determine the result for: 72+44",
 "In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n&%+@] = `?&\n@>*<@ = &>?<\n@`*`% = `^&%\nNow, determine the result for: `]*@>"]

In [82]:
d.answer

2389    7963
3944    4472
6491     <<<
Name: answer, dtype: str

### bit manipulation

In [83]:
bit_df = data[data['label'] == 'bit manipulation'].copy()
solver = BitManipulationSolver()

bit_df['generated_cot'] = bit_df['prompt'].apply(solver.generate_cot)
bit_df['computed_answer'] = bit_df['generated_cot'].apply(solver.extract_answer)
bit_df['is_correct'] = bit_df['computed_answer'].astype(str).str.strip() == bit_df['answer'].astype(str).str.strip()

accuracy = bit_df['is_correct'].mean()
print(f"Accuracy by 'bit manipulation': {accuracy * 100:.2f}%")

Accuracy by 'bit manipulation': 82.96%


In [84]:
errors_df = bit_df[~bit_df['is_correct']]
for idx, row in errors_df.sample(25).iterrows():
    print(f"=== ID: {row['id']} ===")
    print(f"answer:  '{row['answer']}'")
    print(f"Computed:'{row['computed_answer']}'")
    print(f"Prompt: {row['prompt'][-80:]}\n")

=== ID: 7dba5d8b ===
answer:  '11011111'
Computed:'11101111'
Prompt: 001111 -> 11110111
10010111 -> 11101011

Now, determine the output for: 11111010

=== ID: b275443b ===
answer:  '00111000'
Computed:'00111101'
Prompt: 101111 -> 00010111
00011100 -> 10110111

Now, determine the output for: 10100001

=== ID: eb12e80d ===
answer:  '11000111'
Computed:'11010011'
Prompt: 011001 -> 00110001
01110000 -> 11101110

Now, determine the output for: 11101101

=== ID: b265a4af ===
answer:  '00100100'
Computed:'00101100'
Prompt: 011111 -> 00000010
01010110 -> 10100100

Now, determine the output for: 00110110

=== ID: 004ef7c7 ===
answer:  '11111111'
Computed:'11110111'
Prompt: 000010 -> 11111010
00001010 -> 10111011

Now, determine the output for: 01010101

=== ID: 7192535b ===
answer:  '00000010'
Computed:'00001010'
Prompt: 001010 -> 00000100
11000110 -> 10001100

Now, determine the output for: 00100101

=== ID: ada01a20 ===
answer:  '10000000'
Computed:'10000010'
Prompt: 010010 -> 01001010
101110